# Gold: how long the evidence was already there

For every surfaced child, the earliest date their record **already satisfied** the tier
rule — and how long it has sat that way since.

| | |
| --- | --- |
| **Reads** | `gold_observations`, `gold_encounters`, `silver_family_history`, `gold_referral_state` |
| **Writes** | `gold_signal_latency` |

## Why this is the number that matters

The use case is identifying children *early*. A list of surfaced children does not
demonstrate that. This does: it replays each record forward in time and finds the first
date at which the criteria would have fired, had anyone been looking.

> "This child met the multi-system criterion nineteen months ago. Every feature needed
> was already coded. Nobody joined them up."

That is the diagnostic odyssey stated as a measurement rather than an anecdote.

## What it computes, criterion by criterion

Each criterion has a moment it first becomes true, found with a running window over the
record in date order:

| Criterion | First true when |
| --- | --- |
| `MULTI_SYSTEM` | the observation that brings distinct body systems to three |
| `REGRESSION` | developmental regression is first observed |
| `NEURODEV_PLUS` | a neurodevelopmental feature and a second system are both present |
| `DIAGNOSTIC_ODYSSEY` | the encounter at which 4+ specialties, 12+ months and a sub-50% diagnosis rate all hold |
| `REPEAT_UNDIAGNOSED_ADMISSION` | the second admission with no diagnosis recorded |
| `FAMILY_HISTORY` | the date the history was taken, where it recorded something |

The **qualifying date** is then the earliest date the tier rule was met: a sufficient
criterion firing, or a second contributory one joining the first.

## What this does NOT claim

The synthetic record contains no referral events, so this cannot say a referral was
missed or late. It says only: **the evidence has been present and sufficient since this
date.** Whether anyone acted on it is not modelled here, and on real data that is the
comparison you would actually want.

Latency is measured to the observation cutoff, not to "now".

In [ ]:
MIN_SYSTEMS_MULTI = 3
MIN_SPECIALTIES_ODYSSEY = 4
MIN_MONTHS_ODYSSEY = 12
MAX_DIAGNOSED_SHARE_ODYSSEY = 0.5
MIN_UNDIAGNOSED_ADMISSIONS = 2
MIN_CONTRIBUTORY = 2
CUTOFF = "2026-08-27"
PIPELINE_RUN_ID = ""

In [ ]:
import notebookutils
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import DoubleType

RUN_ID = PIPELINE_RUN_ID or "local"
_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table):
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    return spark.read.format("delta").load(
        f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table}")


observations = spark.table("gold_observations")
encounters = spark.table("gold_encounters")
family = lake_table("silver_lakehouse", "silver_family_history")
state = spark.table("gold_referral_state")

surfaced = state.filter("referral_state = 'indicators_present'").select("patient_id")
print(f"surfaced children: {surfaced.count():,}")

In [ ]:
# ------------------------------------ when each phenotype criterion first held
# Running windows over the record in date order. The systems seen so far is a set, so
# its size is the distinct count at that point in time -- which is exactly the quantity
# MULTI_SYSTEM thresholds on.
seen = (Window.partitionBy("patient_id").orderBy("observed_date")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow))

running = (observations.join(surfaced, "patient_id")
           .withColumn("systems_so_far", F.size(F.collect_set("body_system").over(seen)))
           .withColumn("neurodev_so_far",
                       F.max(F.when(F.col("body_system") == "neurodevelopment", 1)
                             .otherwise(0)).over(seen)))

multi_system = (running.filter(F.col("systems_so_far") >= MIN_SYSTEMS_MULTI)
                .groupBy("patient_id")
                .agg(F.min("observed_date").alias("first_true"))
                .withColumn("criterion", F.lit("MULTI_SYSTEM"))
                .withColumn("tier", F.lit("sufficient")))

regression = (observations.join(surfaced, "patient_id")
              .filter(F.col("hpo_id") == "HP:0002376")
              .groupBy("patient_id")
              .agg(F.min("observed_date").alias("first_true"))
              .withColumn("criterion", F.lit("REGRESSION"))
              .withColumn("tier", F.lit("sufficient")))

neurodev_plus = (running.filter((F.col("neurodev_so_far") == 1) &
                                (F.col("systems_so_far") >= 2))
                 .groupBy("patient_id")
                 .agg(F.min("observed_date").alias("first_true"))
                 .withColumn("criterion", F.lit("NEURODEV_PLUS"))
                 .withColumn("tier", F.lit("contributory")))

print(f"MULTI_SYSTEM   first-true dates: {multi_system.count():,}")
print(f"REGRESSION     first-true dates: {regression.count():,}")
print(f"NEURODEV_PLUS  first-true dates: {neurodev_plus.count():,}")

In [ ]:
# ------------------------------------- when each pathway criterion first held
by_date = (Window.partitionBy("patient_id").orderBy("encounter_date")
           .rowsBetween(Window.unboundedPreceding, Window.currentRow))

pathway = (encounters.join(surfaced, "patient_id")
           .withColumn("specialties_so_far",
                       F.size(F.collect_set("specialty").over(by_date)))
           .withColumn("first_encounter",
                       F.min("encounter_date").over(Window.partitionBy("patient_id")))
           .withColumn("months_so_far",
                       F.months_between(F.col("encounter_date"),
                                        F.col("first_encounter")))
           .withColumn("encounters_so_far", F.count("*").over(by_date))
           .withColumn("diagnosed_so_far",
                       F.sum(F.when(F.col("diagnosis_recorded"), 1).otherwise(0))
                       .over(by_date))
           .withColumn("diagnosed_share_so_far",
                       (F.col("diagnosed_so_far") /
                        F.col("encounters_so_far")).cast(DoubleType()))
           .withColumn("undiagnosed_admissions_so_far",
                       F.sum(F.when(F.col("admitted") & ~F.col("diagnosis_recorded"), 1)
                             .otherwise(0)).over(by_date)))

odyssey = (pathway.filter(
    (F.col("specialties_so_far") >= MIN_SPECIALTIES_ODYSSEY) &
    (F.col("months_so_far") >= MIN_MONTHS_ODYSSEY) &
    (F.col("diagnosed_share_so_far") < MAX_DIAGNOSED_SHARE_ODYSSEY))
    .groupBy("patient_id")
    .agg(F.min("encounter_date").alias("first_true"))
    .withColumn("criterion", F.lit("DIAGNOSTIC_ODYSSEY"))
    .withColumn("tier", F.lit("contributory")))

repeat_admission = (pathway.filter(
    F.col("undiagnosed_admissions_so_far") >= MIN_UNDIAGNOSED_ADMISSIONS)
    .groupBy("patient_id")
    .agg(F.min("encounter_date").alias("first_true"))
    .withColumn("criterion", F.lit("REPEAT_UNDIAGNOSED_ADMISSION"))
    .withColumn("tier", F.lit("contributory")))

# Family history is known only from the date it was taken.
family_history = (family.join(surfaced, "patient_id")
                  .filter((F.col("history_taken") == True) &
                          ((F.col("affected_first_degree") == True) |
                           (F.col("consanguinity") == True) |
                           (F.col("recurrent_pregnancy_loss") == True)))
                  .select("patient_id", F.col("asked_on").alias("first_true"))
                  .withColumn("criterion", F.lit("FAMILY_HISTORY"))
                  .withColumn("tier", F.lit("contributory")))

print(f"DIAGNOSTIC_ODYSSEY            {odyssey.count():,}")
print(f"REPEAT_UNDIAGNOSED_ADMISSION  {repeat_admission.count():,}")
print(f"FAMILY_HISTORY                {family_history.count():,}")

In [ ]:
# ------------------------------------------- the date the tier rule was met
first_true = (multi_system.unionByName(regression).unionByName(neurodev_plus)
              .unionByName(odyssey).unionByName(repeat_admission)
              .unionByName(family_history)
              .filter(F.col("first_true").isNotNull()))

first_true.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_criterion_first_true")

# A sufficient criterion qualifies the child the moment it fires.
sufficient_date = (first_true.filter(F.col("tier") == "sufficient")
                   .groupBy("patient_id")
                   .agg(F.min("first_true").alias("sufficient_date")))

# Contributory criteria qualify only in combination: the child is qualified at the
# moment the MIN_CONTRIBUTORY-th one becomes true, which is the Nth earliest date.
ordered = Window.partitionBy("patient_id").orderBy("first_true")
contributory_date = (first_true.filter(F.col("tier") == "contributory")
                     .withColumn("n", F.row_number().over(ordered))
                     .filter(F.col("n") == MIN_CONTRIBUTORY)
                     .select("patient_id",
                             F.col("first_true").alias("contributory_date")))

latency = (surfaced
           .join(sufficient_date, "patient_id", "left")
           .join(contributory_date, "patient_id", "left")
           .withColumn("qualifying_date",
                       F.least(F.col("sufficient_date"), F.col("contributory_date")))
           .withColumn("qualified_by",
                       F.when(F.col("sufficient_date").isNull(), F.lit("combination"))
                        .when(F.col("contributory_date").isNull(), F.lit("sufficient"))
                        .when(F.col("sufficient_date") <= F.col("contributory_date"),
                              F.lit("sufficient"))
                        .otherwise(F.lit("combination")))
           .withColumn("latency_days",
                       F.datediff(F.lit(CUTOFF).cast("date"),
                                  F.col("qualifying_date")))
           .withColumn("latency_months",
                       F.round(F.col("latency_days") / 30.44, 1))
           .withColumn("run_id", F.lit(RUN_ID)))

missing = latency.filter(F.col("qualifying_date").isNull()).count()
print(f"surfaced children with no qualifying date: {missing}")
if missing:
    raise ValueError(
        f"{missing} surfaced children have no date at which their criteria became "
        f"true. Every surfaced child was surfaced by evidence, and all that evidence "
        f"is dated, so a null here means the replay disagrees with the pipeline.")

latency.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_signal_latency")
print(f"gold_signal_latency  {latency.count():,} rows")

In [ ]:
# --------------------------------------------------------------- the headline
latency = spark.table("gold_signal_latency")

stats = latency.select(
    F.count("*").alias("children"),
    F.round(F.avg("latency_months"), 1).alias("mean_months"),
    F.round(F.expr("percentile_approx(latency_months, 0.5)"), 1).alias("median_months"),
    F.round(F.expr("percentile_approx(latency_months, 0.9)"), 1).alias("p90_months"),
    F.max("latency_months").alias("max_months")).collect()[0]

print("How long the qualifying evidence has been in the record")
print(f"  children surfaced   {stats['children']:>6,}")
print(f"  median              {stats['median_months']:>6} months")
print(f"  mean                {stats['mean_months']:>6} months")
print(f"  90th percentile     {stats['p90_months']:>6} months")
print(f"  longest             {stats['max_months']:>6} months")

print("\nBy how the child qualified")
for row in latency.groupBy("qualified_by").agg(
        F.count("*").alias("children"),
        F.round(F.expr("percentile_approx(latency_months, 0.5)"), 1)
         .alias("median_months")).collect():
    print(f"  {row['qualified_by']:14} {row['children']:>5,} children  "
          f"median {row['median_months']} months")

over_year = latency.filter(F.col("latency_months") >= 12).count()
print(f"\nchildren whose evidence has been complete for a year or more: "
      f"{over_year:,} of {stats['children']:,} "
      f"({over_year / stats['children']:.0%})")

print("\nThe ten longest-standing")
for row in (latency.orderBy(F.desc("latency_months"))
            .select("patient_id", "qualifying_date", "qualified_by",
                    "latency_months").limit(10).collect()):
    print(f"  {row['patient_id']}  qualified {row['qualifying_date']}  "
          f"{row['qualified_by']:12} {row['latency_months']:>5} months")

In [ ]:
# ------------------------------- does the delay fall evenly? the harder question
# Being found late is its own harm, separate from not being found at all. The
# sensitivity gap says who gets missed; this says who waits longer among those found.
detail = (spark.table("gold_signal_latency")
          .join(spark.table("gold_referral_state")
                .select("patient_id", "interpreter_required"), "patient_id"))

print("latency by interpreter need")
for row in (detail.groupBy("interpreter_required").agg(
        F.count("*").alias("children"),
        F.round(F.expr("percentile_approx(latency_months, 0.5)"), 1)
         .alias("median_months"),
        F.round(F.avg("latency_months"), 1).alias("mean_months"))
        .orderBy("interpreter_required").collect()):
    key = "interpreter needed" if row["interpreter_required"] else "no interpreter"
    print(f"  {key:20} {row['children']:>5,} children  "
          f"median {row['median_months']:>5} months  mean {row['mean_months']:>5}")

print("\nRead this carefully: it covers only children the screen DID surface. "
      "Children it missed entirely are in gold_validation_sensitivity, and they are "
      "the larger harm.")